## Notebook for using the CDS API to get ERA5 data

#### Run the pip install to get the latest cdsapi

In [1]:
#!pip install cdsapi
!pip install --upgrade cdsapi
# created .cdsapirc file in ~/  per https://ads-beta.atmosphere.copernicus.eu/how-to-api

  Attempting uninstall: cads-api-client
    Found existing installation: cads-api-client 1.0.3
    Uninstalling cads-api-client-1.0.3:
      Successfully uninstalled cads-api-client-1.0.3
  Attempting uninstall: cdsapi
    Found existing installation: cdsapi 0.7.0
    Uninstalling cdsapi-0.7.0:
      Successfully uninstalled cdsapi-0.7.0


In [1]:
import cdsapi
import requests

### This block is for grabbing a single year, entire globe <br><br> This actually errors out and says the request is too large for netcdf, apparently

In [2]:
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            #'2m_temperature', 
            'land_sea_mask',
        ],
        'year': '2023',
        'month': [
            #'01', '02', '03',
            #'04', '05', '06',
            #'07', '08', 
            '09',
            #'10', '11', '12',
        ],
        'day': [
            '01', #'02', '03',
            # '04', '05', '06',
            # '07', '08', '09',
            # '10', '11', '12',
            # '13', '14', '15',
            # '16', '17', '18',
            # '19', '20', '21',
            # '22', '23', '24',
            # '25', '26', '27',
            # '28', '29', '30',
            # '31',
        ],
        'time': [
            # '00:00', '01:00', '02:00',
            # '03:00', '04:00', '05:00',
            # '06:00', '07:00', '08:00',
            # '09:00', '10:00', '11:00',
             '12:00', #'13:00', '14:00',
            # '15:00', '16:00', '17:00',
            # '18:00', '19:00', '20:00',
            # '21:00', '22:00', '23:00',
        ],
        'format': 'netcdf',
    },
    'ERA5-2023-09-01-Full-LSM-download.nc')

2024-07-04 15:47:39,562 INFO Welcome to the CDS
2024-07-04 15:47:39,562 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels
2024-07-04 15:47:39,841 INFO Request is queued
2024-07-04 15:47:42,704 INFO Request is completed
2024-07-04 15:47:42,705 INFO Downloading https://download-0018.copernicus-climate.eu/cache-compute-0018/cache/data5/adaptor.mars.internal-1720133261.933593-1638-18-449eb970-0a03-4b82-b836-fe806558723e.nc to ERA5-2023-09-01-Full-LSM-download.nc (2M)
2024-07-04 15:47:44,709 INFO Download rate 1016.7K/s                            


Result(content_length=2086240,content_type=application/x-netcdf,location=https://download-0018.copernicus-climate.eu/cache-compute-0018/cache/data5/adaptor.mars.internal-1720133261.933593-1638-18-449eb970-0a03-4b82-b836-fe806558723e.nc)

### This block is for specific years and months, specific area lat/lon

In [ ]:
# try multiple years, same region, JJA
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '2m_temperature', 'land_sea_mask',
        ],
        'year': ['2020','2021','2022','2023'],
        'month': [
            '06', '07', '08',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'time': [
            '00:00', '01:00', '02:00',
            '03:00', '04:00', '05:00',
            '06:00', '07:00', '08:00',
            '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00',
            '15:00', '16:00', '17:00',
            '18:00', '19:00', '20:00',
            '21:00', '22:00', '23:00',
        ],
        'area': [
            50, -124, 47,
            -122,
        ],
        'format': 'netcdf',
    },
    '2020-2023_JJA_PugetSound_T2m-LSM.nc')

### <br> Using the Daily Statistics Application to get daily means instead of pulling hourly data and calculating it <br> https://cds.climate.copernicus.eu/cdsapp#!/software/app-c3s-daily-era5-statistics?tab=overview <br><br> From this forum post: https://forum.ecmwf.int/t/retrieve-daily-era5-era5-land-data-using-the-cds-api/1150/1 <br><br> After running - note each month is roughly 128MB or about 1.5GB per year! <br> 

In [3]:
import cdsapi
import requests
 
# CDS API script to use CDS service to retrieve daily ERA5* variables and iterate over
# all months in the specified years.
 
# Requires:
# 1) the CDS API to be installed and working on your system
# 2) You have agreed to the ERA5 Licence (via the CDS web page)
# 3) Selection of required variable, daily statistic, etc
 
# Output:
# 1) separate netCDF file for chosen daily statistic/variable for each month
 
c = cdsapi.Client()#timeout=300)
 
# Uncomment years as required
 
years =  [
#    '1960',
#    '1961','1962','1963','1964'
#   '1965','1966',
#    '1967',
#    '1968',
#     '1969',
#            '1970',
#    '1971','1972','1973','1974','1975',
#    '1976','1977','1978','1979',
#            '1980', 
#    '1981','1982', '1983', '1984',
#            '1985',
# '1986', '1987','1988', '1989', 
#    '1990'#,
  #          '1991', '1992', '1993',
  #          '1994', '1995', '1996',
  #          '1997', '1998', '1999',
  #         '2000',
  #  '2001', '2002',
  #          '2003', '2004', '2005',
  #          '2006', '2007', '2008',
  #          '2009', 
  # '2010', 
   # '2011',
   #          '2012', '2013', '2014',
   #          '2015', '2016', '2017',
   #          '2018', '2019', '2020',
   #         '2021',
   #          '2022',
   # '2023',
   #  '2024'
]
 
 
# Retrieve all months for a given year.
 
months = ['01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12']
 
# For valid keywords, see Table 2 of:
# https://datastore.copernicus-climate.eu/documents/app-c3s-daily-era5-statistics/C3S_Application-Documentation_ERA5-daily-statistics-v2.pdf
 
# select your variable; name must be a valid ERA5 CDS API name.
var = "sea_surface_temperature" #"2m_temperature"
 
# Select the required statistic, valid names given in link above
stat = "daily_mean"
 
# Loop over years and months
 
for yr in years:
    for mn in months:
        result = c.service(
        "tool.toolbox.orchestrator.workflow",
        params={
             "realm": "user-apps",
             "project": "app-c3s-daily-era5-statistics",
             "version": "master",
             "kwargs": {
                 "dataset": "reanalysis-era5-single-levels",
                 "product_type": "reanalysis",
                 "variable": var,
                 "statistic": stat,
                 "year": yr,
                 "month": mn,
                 "time_zone": "UTC+00:0",
                 "frequency": "1-hourly",
#
# Users can change the output grid resolution and selected area
#
#                "grid": "1.0/1.0",
#                "area":{"lat": [10, 60], "lon": [65, 140]}
 
                 },
        "workflow_name": "application"
        })
         
# set name of output file for each month (statistic, variable, year, month     
 
        file_name = "../../../Data/ERA5-global/Analysis/" + yr + "/download_" + stat + "_" + var + "_" + yr + "_" + mn + ".nc"
         
        location=result[0]['location']
        res = requests.get(location, stream = True)
        print("Writing data to " + file_name)
        with open(file_name,'wb') as fh:
            for r in res.iter_content(chunk_size = 1024):
                fh.write(r)
        fh.close()

2024-09-01 15:23:56,295 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-09-01 15:23:56,297 WARNING MOVE TO CDS-Beta
2024-09-01 15:23:56,297 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-4b6785b37c5d4dba94e39039ce3762ea
2024-09-01 15:23:56,617 INFO Request is queued
2024-09-01 15:23:57,857 INFO Request is running
2024-09-01 15:26:50,012 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Analysis/2011/download_daily_mean_sea_surface_temperature_2011_01.nc


2024-09-01 15:27:31,386 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-09-01 15:27:31,387 WARNING MOVE TO CDS-Beta
2024-09-01 15:27:31,388 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-45aedb1d085540979a832cc351cee016
2024-09-01 15:27:31,620 INFO Request is queued
2024-09-01 15:27:32,835 INFO Request is running
2024-09-01 15:30:25,176 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Analysis/2011/download_daily_mean_sea_surface_temperature_2011_02.nc


2024-09-01 15:30:35,986 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-09-01 15:30:35,988 WARNING MOVE TO CDS-Beta
2024-09-01 15:30:35,989 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-e474a899fab24c8a9e99bbcf8f8eadac
2024-09-01 15:30:36,257 INFO Request is queued
2024-09-01 15:30:37,468 INFO Request is running
2024-09-01 15:30:39,182 INFO Request is completed


...

Writing data to ../../../Data/ERA5-global/Analysis/2024/download_daily_mean_sea_surface_temperature_2024_07.nc


2024-09-01 22:23:36,166 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-09-01 22:23:36,166 WARNING MOVE TO CDS-Beta
2024-09-01 22:23:36,166 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-2674b273e7ce49c38b86417134c94dc0
2024-09-01 22:23:36,410 INFO Request is queued
2024-09-01 22:23:37,639 INFO Request is running
2024-09-01 22:27:57,302 INFO Request is completed


Writing data to ../../../Data/ERA5-global/Analysis/2024/download_daily_mean_sea_surface_temperature_2024_08.nc


2024-09-01 22:28:48,084 INFO Welcome to the CDS.
 As per our announcements on the Forum, this instance of CDS will soon be decommissioned.
 Please update your cdsapi package to a version >=0.7.0, create an account on CDS-Beta and update your .cdsapirc file. We strongly recommend users to check our Guidelines at https://confluence.ecmwf.int/x/uINmFw
 The current legacy system will be kept for a while, but we will reduce resources gradually until full decommissioning in September 2024.
2024-09-01 22:28:48,084 WARNING MOVE TO CDS-Beta
2024-09-01 22:28:48,085 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-1114a7e9269d40b68b4fe4d047327e39
2024-09-01 22:28:48,305 INFO Request is queued
2024-09-01 22:28:49,505 INFO Request is running
2024-09-01 22:28:51,226 INFO Request is failed
2024-09-01 22:28:51,226 ERROR Message: 
2024-09-01 22:28:51,227 ERROR Reason:  Traceback (most recent call last):
  File "/opt/cdstoolbox/c

Exception: . Traceback (most recent call last):
  File "/opt/cdstoolbox/cdscompute/cdscompute/cdshandlers/services/handler.py", line 59, in handle_request
    result = cached(context.method, proc, context, context.args, context.kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/caching.py", line 108, in cached
    result = proc(context, *context.args, **context.kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/services.py", line 124, in __call__
    return p(*args, **kwargs)
  File "/opt/cdstoolbox/cdscompute/cdscompute/services.py", line 60, in __call__
    return self.proc(context, *args, **kwargs)
  File "/home/cds/cdsservices/services/retrieve.py", line 197, in execute
    remote = context.call_resource(name, request, update_specific_metadata={'app_scope': 'adaptor'})
  File "/opt/cdstoolbox/cdscompute/cdscompute/context.py", line 307, in call_resource
    return c.call_resource(service, *args, **kwargs).value
  File "/opt/cdstoolbox/cdsworkflows/cdsworkflows/future.py", lin

## 

In [4]:
!ls -al "../../../Data/ERA5-global/Analysis/2024/"

total 3578240
drwxr-xr-x@ 17 tedscott  staff        544 Sep  1 22:28 .
drwxr-xr-x@ 39 tedscott  staff       1248 Aug 30 11:36 ..
-rw-r--r--@  1 tedscott  staff  128778531 Jul 23 15:15 download_daily_mean_2m_temperature_2024_01.nc
-rw-r--r--@  1 tedscott  staff  120472595 Jul 23 15:18 download_daily_mean_2m_temperature_2024_02.nc
-rw-r--r--@  1 tedscott  staff  128778532 Jul 23 15:22 download_daily_mean_2m_temperature_2024_03.nc
-rw-r--r--@  1 tedscott  staff  124625564 Jul 23 15:23 download_daily_mean_2m_temperature_2024_04.nc
-rw-r--r--@  1 tedscott  staff  128778534 Jul 23 15:23 download_daily_mean_2m_temperature_2024_05.nc
-rw-r--r--@  1 tedscott  staff  124625566 Jul 23 15:23 download_daily_mean_2m_temperature_2024_06.nc
-rw-r--r--@  1 tedscott  staff   74789948 Jul 23 15:23 download_daily_mean_2m_temperature_2024_07.nc
-rw-r--r--@  1 tedscott  staff  128778531 Sep  1 22:02 download_daily_mean_sea_surface_temperature_2024_01.nc
-rw-r--r--@  1 tedscott  staff  120472597 Sep  1 22:06

## Get Satellite Ocean Color Data for Salome and Risako

In [2]:
import cdsapi
import requests

# try multiple years, specific region including Galapagos
c = cdsapi.Client()

c.retrieve(
    'satellite-ocean-colour',
    {
        'version': '6_0',
        'format': 'zip',
        'variable': [
            'mass_concentration_of_chlorophyll_a',
        ],
        'projection': 'regular_latitude_longitude_grid',
        'year': ['2019',
                 #'2020','2021','2022','2023',
                ],
        'month': [
            '01', '02', '03','04','05','06','07','08','09','10','11','12',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'area': [
            2, -92, -2,
            -88,
        ],
        'format': 'netcdf',
    },
    '2019-2023_OceanColour.nc')

2024-07-09 15:25:15,108 INFO Welcome to the CDS
2024-07-09 15:25:15,108 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/satellite-ocean-colour
2024-07-09 15:25:15,301 INFO Request is queued
2024-07-09 15:25:16,480 INFO Request is failed
2024-07-09 15:25:16,481 ERROR Message: the request you have submitted is not valid
2024-07-09 15:25:16,481 ERROR Reason:  Request too large. Requesting 90 items, limit is 31
2024-07-09 15:25:16,482 ERROR   Traceback (most recent call last):
2024-07-09 15:25:16,483 ERROR     File "/opt/cds/cdsinf/python/lib/cdsinf/runner/dispatcher.py", line 163, in _consume
2024-07-09 15:25:16,483 ERROR       result = handle_locally()
2024-07-09 15:25:16,483 ERROR     File "/opt/cds/cdsinf/python/lib/cdsinf/runner/dispatcher.py", line 252, in <lambda>
2024-07-09 15:25:16,484 ERROR       lambda: self.handle_exception(context, e),
2024-07-09 15:25:16,484 ERROR     File "/opt/cds/cdsinf/python/lib/cdsinf/runner/dispatcher.py", line 383, in handle

Exception: the request you have submitted is not valid. Request too large. Requesting 90 items, limit is 31.